In [0]:
import pandas as pd

base_path = "/tmp/healthcare_project_raw"

providers = pd.read_csv(f"{base_path}/providers.csv")
members = pd.read_csv(f"{base_path}/members.csv")
claims = pd.read_csv(f"{base_path}/claims.csv")

print("Providers:", providers.shape)
print("Members:", members.shape)
print("Claims:", claims.shape)

In [0]:
print("Providers")
display(providers.head())

print("Members")
display(members.head())

print("Claims")
display(claims.head())

In [0]:
print("Duplicate claim IDs:")
display(
    claims[claims.duplicated("claim_id", keep=False)]
)

# print("Missing values in claims:")
# display(
#     claims.isna().sum()
# )

print("Invalid claim amounts:")
display(
    claims[claims["claim_amount"] <= 0]
)

print("Invalid member references:")
display(
    claims[~claims["member_id"].isin(members["member_id"])]
)

print("Invalid provider references:")
display(
    claims[~claims["provider_id"].isin(providers["provider_id"])]
)

print("Member regions:")
display(
    members["region"].value_counts()
)

In [0]:
claims[claims.duplicated("claim_id", keep=False)]

In [0]:
import shutil

source = "/tmp/healthcare_project_raw"
destination = "/Workspace/Users/durga20010615@gmail.com/healthcare-data-engineering-pipeline/data/raw"

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Files copied.")

In [0]:
spark.table("bronze_claims").count()

In [0]:
spark.table("bronze_claims").count()

In [0]:
from pathlib import Path

incoming_dir = Path(
    "/Workspace/Users/durga20010615@gmail.com/healthcare-data-engineering-pipeline/data/incoming"
)

incoming_files = sorted(incoming_dir.glob("claims_batch_*.csv"))

print("Incoming files:", [f.name for f in incoming_files])


In [0]:

if incoming_files:
    claims_df = spark.read.option("header", True).csv(
        [str(file) for file in incoming_files]
    )
    print("Incoming claims:", claims_df.count())
else:
    print("No incoming claim files found")


In [0]:
from pyspark.sql import functions as F

bronze = spark.table("bronze_claims")

# Check duplicate claim IDs
display(
    bronze.groupBy("claim_id")
    .count()
    .filter(F.col("count") > 1)
)

# Check batch 3 IDs in Bronze
display(
    bronze.filter(
        F.col("claim_id").between("CLM00331", "CLM00340")
    ).orderBy("claim_id")
)

print("Total Bronze rows:", bronze.count())
print("Unique claim IDs:", bronze.select("claim_id").distinct().count())

In [0]:
from pyspark.sql import functions as F

claims = spark.table("silver_claims")

# Check duplicate claim IDs
claims.groupBy("claim_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

# Inspect claims with missing status
claims.filter(F.col("claim_status").isNull()).show(truncate=False)

# Check Bronze total versus unique claim IDs
bronze = spark.table("bronze_claims")

print("Bronze rows:", bronze.count())
print("Unique Bronze claim IDs:",
      bronze.select("claim_id").distinct().count())